# Sentinel-2 Patch & Ground Truth Visualizer (RPG 2022)
Multi-temporal Sentinel-2 `.pt` patch cubes and paired 19-class French RPG 2022 Cadastre crop masks.

In [ ]:
# Imports. Check PyTorch and Matplotlib.
import os
import glob
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

plt.rcParams['figure.dpi'] = 120
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")


In [ ]:
# PASTIS-24 normalization constants (10 bands).
S2_MEAN = torch.tensor([
    1165.94, 1375.65, 1429.22, 1764.80, 2719.27,
    3063.61, 3205.90, 3319.11, 2422.90, 1639.37
], dtype=torch.float32).view(1, 10, 1, 1)

S2_STD = torch.tensor([
    1942.62, 1881.92, 1959.38, 1867.22, 1754.59,
    1769.40, 1784.86, 1767.71, 1458.96, 1299.28
], dtype=torch.float32).view(1, 10, 1, 1)

# Official 19 crop classes from French RPG 2022 Cadastre.
CROP_CLASSES = {
    0: "Non-agri / Road / Hedge",
    1: "Permanent meadow",
    2: "Soft winter wheat",
    3: "Corn / Maize",
    4: "Winter barley",
    5: "Winter rapeseed",
    6: "Spring barley",
    7: "Sunflower",
    8: "Soybean",
    9: "Peas",
    10: "Protein crops",
    11: "Other oilseeds",
    12: "Vegetables / Flowers",
    13: "Vineyard",
    14: "Fruit orchards (Apples)",
    15: "Fallow",
    16: "Potato",
    17: "Sugar beet",
    18: "Other cereals"
}

# Distinct colors for 19 classes.
CROP_COLORS = [
    '#343a40', '#2ca02c', '#f5b041', '#e67e22', '#d68910',
    '#27ae60', '#f39c12', '#f1c40f', '#1abc9c', '#16a085',
    '#3498db', '#2980b9', '#10b981', '#8e44ad', '#9b59b6',
    '#a04000', '#d35400', '#e74c3c', '#8000ff'
]
CMAP_CROP = ListedColormap(CROP_COLORS)
NORM_CROP = BoundaryNorm(boundaries=np.arange(len(CROP_COLORS) + 1) - 0.5, ncolors=len(CROP_COLORS))


In [ ]:
# Locate .pt patches in Kaggle input or local disk.
def locate_dataset():
    # Search /kaggle/input first.
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            pts = [f for f in files if f.endswith(".pt")]
            if pts:
                print(f"Kaggle: Found {len(pts)} patches in {root}")
                return {f: os.path.join(root, f) for f in pts}

    # Fallback to local paths.
    search_paths = [
        "./crop_classification_tsvit_dinov3/data/processed/patches",
        "../crop_classification_tsvit_dinov3/data/processed/patches",
        "./data/processed/patches",
        "."
    ]
    for base in search_paths:
        if os.path.exists(base):
            pts = glob.glob(os.path.join(base, "**/*.pt"), recursive=True)
            if pts:
                print(f"Local: Found {len(pts)} patches in {base}")
                return {os.path.basename(p): p for p in pts}

    raise FileNotFoundError("No .pt patch files found.")

PATCH_MAP = locate_dataset()


## 1. Helper Functions: Contrast & 4-Panel Visualization

In [ ]:
# Denormalize S2 reflectance.
def denormalize(tensor_10b):
    return tensor_10b * S2_STD + S2_MEAN

# Pick clearest cloud-free acquisition index.
def select_best_t(raw_cube):
    scores = []
    for t in range(raw_cube.shape[0]):
        nir = raw_cube[t, 6]
        red = raw_cube[t, 2]
        blue = raw_cube[t, 0]
        # Skip cloudy or corrupt frames.
        if red.mean() < 150 or red.mean() > 4500 or blue.mean() > 3500:
            scores.append(-999)
            continue
        # Rank by NDVI contrast and standard deviation.
        ndvi = (nir - red) / (nir + red + 1e-6)
        score = ndvi.mean().item() * 0.7 + (raw_cube[t, [2, 1, 0]].std().item() / 1000.0) * 0.3
        scores.append(score)
    return int(np.argmax(scores))

# Contrast stretch (percentile 2% - 98%).
def stretch_contrast(arr_3ch):
    p2, p98 = np.percentile(arr_3ch, (2, 98))
    res = np.clip((arr_3ch - p2) / (p98 - p2 + 1e-6), 0.0, 1.0)
    return np.power(res, 1.0 / 1.1)

# Render single patch with 4 panels: True Color, False Color NIR, Ground Truth, Superimposed Overlay.
def visualize_patch(patch_name, t_index=None):
    path = PATCH_MAP[patch_name]
    d = torch.load(path, map_location='cpu', weights_only=False)

    raw = denormalize(d['image'])
    if t_index is None:
        t_index = select_best_t(raw)
    doy = int(round(d['doy'][t_index].item() * 365.0))

    rgb = stretch_contrast(raw[t_index, [2, 1, 0]].permute(1, 2, 0).numpy())
    cir = stretch_contrast(raw[t_index, [6, 2, 1]].permute(1, 2, 0).numpy())

    # Load ground truth mask.
    label_key = 'label' if 'label' in d else ('mask' if 'mask' in d else None)
    gt = d[label_key].numpy() if label_key else np.zeros((24, 24), dtype=np.int64)

    # Build legend for classes present in this patch.
    present_classes = np.unique(gt)
    legend_patches = [
        Patch(facecolor=CROP_COLORS[c], edgecolor='black', label=f"[{c}] {CROP_CLASSES.get(c, 'Unknown')}")
        for c in present_classes
    ]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

    # 1. True Color RGB.
    axes[0].imshow(rgb, interpolation='nearest')
    axes[0].set_title(f"True Color RGB (10m)\n{patch_name} (DOY {doy})", fontsize=10, fontweight='bold')
    axes[0].axis('off')

    # 2. False Color CIR (NIR-R-G).
    axes[1].imshow(cir, interpolation='nearest')
    axes[1].set_title("False Color CIR (NIR)\nRed = Healthy Vegetation", fontsize=10, fontweight='bold')
    axes[1].axis('off')

    # 3. Ground Truth RPG 2022.
    axes[2].imshow(gt, cmap=CMAP_CROP, norm=NORM_CROP, interpolation='nearest')
    axes[2].set_title("Ground Truth RPG 2022\nCadastre Mask (UTM 31N)", fontsize=10, fontweight='bold')
    axes[2].axis('off')

    # 4. Superimposed Overlay with parcel boundaries.
    axes[3].imshow(rgb, interpolation='nearest')
    axes[3].contour(gt, levels=np.arange(0.5, len(CROP_COLORS)), colors='black', linewidths=1.8)
    axes[3].imshow(gt, cmap=CMAP_CROP, norm=NORM_CROP, alpha=0.35, interpolation='nearest')
    axes[3].set_title("Superimposed Overlay\nBlack = Parcel Edges", fontsize=10, fontweight='bold')
    axes[3].axis('off')

    # Place legend on right side.
    axes[3].legend(handles=legend_patches, bbox_to_anchor=(1.05, 1.0), loc='upper left', frameon=True, fontsize=9)
    plt.tight_layout()
    plt.show()


## 2. Single Patch Inspection (`patch_00013.pt`)

In [ ]:
# Test single patch with 4 panels.
sample_patch = 'patch_00013.pt' if 'patch_00013.pt' in PATCH_MAP else list(PATCH_MAP.keys())[0]
visualize_patch(sample_patch)


## 3. 10 Verified Representative Patches & Ground Truth
Inspect 10 verified agricultural patches with Cadastre field boundary contours.

In [ ]:
# 10 verified agricultural patches.
TEN_VERIFIED_PATCHES = [
    'patch_00233.pt',
    'patch_00583.pt',
    'patch_00597.pt',
    'patch_00013.pt',
    'patch_00028.pt',
    'patch_00387.pt',
    'patch_00401.pt',
    'patch_00027.pt',
    'patch_00604.pt',
    'patch_00779.pt'
]

# Filter available patches. Fallback to first available if missing.
to_display = [p for p in TEN_VERIFIED_PATCHES if p in PATCH_MAP]
if len(to_display) < 10:
    for p in PATCH_MAP:
        if p not in to_display and len(to_display) < 10:
            to_display.append(p)

print(f"Visualizing {len(to_display)} patches:")
for idx, p in enumerate(to_display, 1):
    print(f"==================================================")
    print(f"Patch #{idx:02d}: {p}")
    print(f"==================================================")
    visualize_patch(p)


## 4. Crop Class Distribution in Inspected Patches

In [ ]:
# Print crop breakdown for each patch.
print(f"{'#':<4} | {'Patch':<16} | {'Crops Present'}")
print("-" * 75)
for idx, p in enumerate(to_display, 1):
    d = torch.load(PATCH_MAP[p], map_location='cpu', weights_only=False)
    label_key = 'label' if 'label' in d else ('mask' if 'mask' in d else None)
    gt = d[label_key].numpy() if label_key else np.zeros((24, 24), dtype=np.int64)
    classes = [f"[{c}] {CROP_CLASSES.get(c, 'Unknown')}" for c in np.unique(gt)]
    print(f"{idx:<4} | {p:<16} | {', '.join(classes)}")
